# Notebook 42 — MLP input layer: covered features or live units?

Notebook 35 showed the MLP collapses when its input layer is starved to 38 surviving weights. But an MLP input
weight connects one feature to one unit, so starvation can act through two routes: fewer live units (channel
starvation, the CNN mechanism) or fewer covered features (feature dropout). Under magnitude pruning the two
move together. This notebook separates them by construction.

**Design.** On the five MLP baselines (`M0`, seeds 0-4; hidden 256/128), the input layer (256 units x 39 features)
receives a hand-built mask with exactly T surviving weights arranged as a K-unit by F-feature block (K x F = T),
choosing the K units with the largest row magnitude and the F features with the largest column magnitude.
At T = 96: (K, F) in {(96, 1), (32, 3), (12, 8), (8, 12), (3, 32)}; at T = 192: {(192, 1), (64, 3), (24, 8), (16, 12), (6, 32)}.
The second hidden layer and head are pruned at 80% by magnitude. Live units and covered features are asserted
after fine-tuning.

**Gate (stated before running).** At fixed T, if damage is governed by covered features, loss falls as F rises
even while K falls (Spearman between F and loss <= -0.8 within each T); if by live units, loss falls as K rises
(Spearman between K and loss <= -0.8). Because K and F are inversely tied at fixed T, exactly one of these can
hold. Any outcome is reported. Resumable per (seed, T, K). GPU runtime required.

In [ ]:
# --- Colab bootstrap ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys, copy, json as _json
os.chdir(REPO); sys.path.insert(0, REPO)
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.utils.prune as prune
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from src.config import CFG, PATHS, set_all_seeds
from src.data import load_raw, clean, temporal_within_capture_split
from src import train as TR, models as M, explain as EXP, mitigate
from src.comnet_audit import assign_validation_tiers, calibration_summary, environment_record, write_json
from src.train import load_anchor, predict, per_class_recall_table, feature_columns

assert torch.cuda.is_available(), 'switch to a GPU runtime first'
DEVICE = TR.DEVICE
DATASET = 'ciciot2023'; ARCH = 'mlp'
SEEDS = list(CFG['seeds']); ANCHOR = int(CFG['anchor_seed'])
OUT = PATHS.tables('comnet')
PRACTICAL_LOSS = 0.10
GRID = {96: [(96, 1), (32, 3), (12, 8), (8, 12), (3, 32)], 192: [(192, 1), (64, 3), (24, 8), (16, 12), (6, 32)]}

ARCH_KW = {'hidden': (256, 128)}; BASE_CELL = 'M0'
from scipy.stats import spearmanr
for T, g in GRID.items():
    for K, F in g: assert K * F == T, (T, K, F)
print('grid verified:', {T: [(K, F) for K, F in g] for T, g in GRID.items()})

In [ ]:
df = clean(load_raw(DATASET, subsample=True, seed=ANCHOR), DATASET)
splits = temporal_within_capture_split(df, seed=ANCHOR)
feat_cols = feature_columns(df)
print(f'{len(df):,} rows | {df.label.nunique()} classes')

In [ ]:
# Shared helpers (as in Notebooks 34-38) plus the block-mask constructor for the MLP input layer.
def prunable(model):
    return [(mod, 'weight') for mod in model.modules() if isinstance(mod, (nn.Linear, nn.Conv1d))]

def layer_names(model):
    return {mod: n for n, mod in model.named_modules()}

def layer_sparsity(model):
    names = layer_names(model); out = {}
    for mod, name in prunable(model):
        w = getattr(mod, name); out[names[mod]] = float((w == 0).float().mean())
    z = sum(int((getattr(mod, n) == 0).sum()) for mod, n in prunable(model)); n_ = sum(getattr(mod, n).numel() for mod, n in prunable(model))
    out['prunable_sparsity'] = z / n_; out['remaining_nonzero_prunable'] = n_ - z
    return out

def finetune_masked(model, seed, *, ft_epochs=8, batch_size=4096, lr=5e-4, verbose=False):
    set_all_seeds(seed)
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    le = LabelEncoder().fit(df['label'].to_numpy())
    scaler = StandardScaler().fit(df.loc[splits['train'], feat_cols].to_numpy(np.float32))
    t = TR.make_tensors(df, splits, feat_cols, le, scaler); Xtr, ytr = t['train']
    model = model.to(DEVICE)
    masks = {(mod, name): (getattr(mod, name) != 0).float() for mod, name in prunable(model)}
    hooks = [getattr(mod, name).register_hook((lambda mk: (lambda g: g * mk))(mk)) for (mod, name), mk in masks.items()]
    w = TR.tempered_class_weights(ytr.numpy(), len(le.classes_))
    crit = nn.CrossEntropyLoss(weight=w); opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    for ep in range(ft_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
        if verbose: print(f'    ft epoch {ep}')
    for h in hooks: h.remove()
    with torch.no_grad():
        for mod, name in prunable(model): getattr(mod, name).mul_((getattr(mod, name) != 0).float())
    return model.eval(), le, scaler

def save_ckpt(model, le, scaler, path):
    torch.save({'state_dict': model.state_dict(), 'classes': list(le.classes_), 'feat_cols': feat_cols,
                'scaler_mean': scaler.mean_, 'scaler_scale': scaler.scale_}, path)


def block_mask(w, K, F):
    # w: (units=256, features=39). Keep a K x F block: top-K units by row L1, top-F features by column L1.
    a = w.abs(); units = a.sum(dim=1).argsort(descending=True)[:K]; feats = a.sum(dim=0).argsort(descending=True)[:F]
    mask = torch.zeros_like(w, dtype=torch.bool); mask[units.unsqueeze(1), feats.unsqueeze(0)] = True
    assert int(mask.sum()) == K * F, (int(mask.sum()), K, F); return mask

def apply_grid(model, K, F):
    m = copy.deepcopy(model); names = layer_names(m)
    inp = [mod for mod, _ in prunable(m) if names[mod] == 'body.0'][0]
    with torch.no_grad(): inp.weight.mul_(block_mask(inp.weight.detach().cpu(), K, F).to(inp.weight.device).float())
    for mod, name in prunable(m):
        if names[mod] == 'body.0': continue
        prune.l1_unstructured(mod, name=name, amount=0.8); prune.remove(mod, name)
    return m

def input_stats(model):
    names = layer_names(model); inp = [mod for mod, _ in prunable(model) if names[mod] == 'body.0'][0]
    nz = (inp.weight.detach() != 0); return int(nz.any(dim=1).sum()), int(nz.any(dim=0).sum()), int(nz.sum())

m_chk, _, _, _ = load_anchor(DATASET, ARCH, BASE_CELL, ANCHOR, arch_kwargs=ARCH_KW)
assert [layer_names(m_chk)[mod] for mod, _ in prunable(m_chk)] == ['body.0', 'body.3', 'head']
w0 = [mod for mod, _ in prunable(m_chk) if layer_names(m_chk)[mod] == 'body.0'][0].weight.detach().cpu()
assert w0.shape == (256, len(feat_cols)), w0.shape
for T, g in GRID.items():
    for K, F in g: mk = block_mask(w0, K, F); print(f'  T={T:3d} K={K:3d} F={F:2d}: live units {int(mk.any(dim=1).sum())}, covered features {int(mk.any(dim=0).sum())}')

In [ ]:
# Run the grid on five baselines, with resume
baseline_val, baseline_test, comp_test, macro, stat_rows = {}, {}, {}, [], []
for seed in SEEDS:
    print(f'\n===== seed {seed} =====')
    m0, le, scaler, _ = load_anchor(DATASET, ARCH, BASE_CELL, seed, arch_kwargs=ARCH_KW)
    yv, pv, _ = predict(m0, df, splits, le, scaler, feat_cols, which='val'); yt, pt, _ = predict(m0, df, splits, le, scaler, feat_cols, which='test')
    baseline_val[seed] = per_class_recall_table(yv, pv, le).set_index('label')['recall']; baseline_test[seed] = per_class_recall_table(yt, pt, le).set_index('label')['recall']
    macro.append({'seed': seed, 'T': 0, 'K': 256, 'F': len(feat_cols), 'cell': 'M0', 'test_macro_f1': f1_score(yt, pt, average='macro')})
    for T, g in GRID.items():
        for K, F in g:
            cell = f'grid{T}_K{K}_F{F}_mlp_paired'; p_c = PATHS.model(DATASET, ARCH, cell, seed)
            if os.path.exists(p_c):
                mp = M.build(ARCH, len(feat_cols), len(le.classes_), **ARCH_KW).to(DEVICE)
                mp.load_state_dict(torch.load(p_c, map_location=DEVICE, weights_only=False)['state_dict']); mp.eval(); print(f'  loaded {cell}')
            else:
                mp, _, _ = finetune_masked(apply_grid(m0, K, F), seed); save_ckpt(mp, le, scaler, p_c); print(f'  saved {cell}')
            live, covered, taps = input_stats(mp); assert (live, covered, taps) == (K, F, T), (cell, seed, live, covered, taps)
            stat_rows.append({'seed': seed, 'T': T, 'K': K, 'F': F, 'cell': cell, 'live_units': live, 'covered_features': covered, 'surviving_input_weights': taps})
            yt, pc, _ = predict(mp, df, splits, le, scaler, feat_cols, which='test')
            comp_test[(seed, T, K, F)] = per_class_recall_table(yt, pc, le).set_index('label')['recall']
            macro.append({'seed': seed, 'T': T, 'K': K, 'F': F, 'cell': cell, 'test_macro_f1': f1_score(yt, pc, average='macro')})
print('\ngrid complete')

In [ ]:
# Aggregate + gate
tiers = assign_validation_tiers(pd.DataFrame(baseline_val)); rows = []
for (seed, T, K, F), rc in comp_test.items():
    r0 = baseline_test[seed]
    for cls in r0.index.intersection(rc.index):
        loss = float(r0.loc[cls] - rc.loc[cls]); band = float(tiers.loc[cls, 'validation_2sd_band'])
        rows.append({'seed': seed, 'T': T, 'K': K, 'F': F, 'class': cls, 'recall_loss': loss, 'material_and_beyond_band': bool((loss > band) and (loss >= PRACTICAL_LOSS))})
eff = pd.DataFrame(rows); assert len(eff) > 0, 'no rows: run the grid cell in this session first'
eff.to_csv(OUT / 'mlp_grid_per_class_effects.csv', index=False)
summ = eff.groupby(['T', 'K', 'F', 'class']).agg(mean_recall_loss=('recall_loss', 'mean'), affected_frequency=('material_and_beyond_band', 'mean')).reset_index(); summ.to_csv(OUT / 'mlp_grid_per_class_summary.csv', index=False)
mdf = pd.DataFrame(macro); mdf.to_csv(OUT / 'mlp_grid_macro_f1_wide.csv', index=False); pd.DataFrame(stat_rows).to_csv(OUT / 'mlp_grid_input_stats.csv', index=False)
m0m = mdf[mdf.cell == 'M0'].test_macro_f1.mean(); tab = []
for T, g in GRID.items():
    for K, F in g:
        s = mdf[(mdf['T'] == T) & (mdf.K == K) & (mdf.F == F)].test_macro_f1
        tab.append({'T': T, 'live_units_K': K, 'covered_features_F': F, 'mean_macro_f1': s.mean(), 'sd': s.std(), 'mean_macro_f1_loss': m0m - s.mean(),
                    'classes_affected_ge3of5': int((summ[(summ['T'] == T) & (summ.K == K) & (summ.F == F)].affected_frequency >= 0.6).sum())})
tab = pd.DataFrame(tab); tab.to_csv(OUT / 'mlp_grid_comparison.csv', index=False)
print(f'M0 five-seed macro-F1: {m0m:.4f}\n'); print(tab.round(4).to_string(index=False))
ver = []
for T in GRID:
    t = tab[tab['T'] == T]; rF = spearmanr(t.covered_features_F, t.mean_macro_f1_loss).correlation; rK = spearmanr(t.live_units_K, t.mean_macro_f1_loss).correlation
    ver.append({'T': T, 'spearman_features_vs_loss': round(float(rF), 3), 'spearman_units_vs_loss': round(float(rK), 3),
                'features_govern': bool(rF <= -0.8), 'units_govern': bool(rK <= -0.8)})
verdict = pd.DataFrame(ver); print(); print(verdict.to_string(index=False))
print('\nMLP damage governed by covered FEATURES at both T:', bool(verdict.features_govern.all()), '| by live UNITS at both T:', bool(verdict.units_govern.all()))
verdict.to_csv(OUT / 'mlp_grid_gate_verdict.csv', index=False)
write_json(OUT / 'mlp_grid_environment.json', {'grid': {str(k): v for k, v in GRID.items()}, 'seeds': SEEDS, 'environment': environment_record()})

In [ ]:
# --- Commit + push: main only, this notebook's own files only ---
import subprocess, shutil, glob
_b = subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert _b == 'main', f'checked-out branch is {_b!r}; run `git checkout main` first'
subprocess.run(['git', 'config', '--global', 'user.name', 'Md Anas Biswas'], check=True)
subprocess.run(['git', 'config', '--global', 'user.email', 'anasbiswas@gmail.com'], check=True)
cred = '/content/drive/MyDrive/IoT_Trust_Research/.git-credentials'
if os.path.exists(cred):
    shutil.copy(cred, '/root/.git-credentials'); subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
_own = 'notebooks/42_mlp_feature_vs_unit_grid.ipynb'
if os.path.exists(_own):
    d = _json.load(open(_own))
    for c in d.get('cells', []):
        if c.get('cell_type') == 'code': c['outputs'] = []; c['execution_count'] = None
    _json.dump(d, open(_own, 'w'), indent=1)
subprocess.run(['git', 'add', _own] + glob.glob('results/tables/comnet/mlp_grid_*'), check=True)
r = subprocess.run(['git', 'commit', '-m', 'notebook 42: MLP input layer at fixed weight count - covered features vs live units grid (T=96, 192), five baselines; mechanism-separation gate'], capture_output=True, text=True)
print(r.stdout or r.stderr)
print(subprocess.run(['git', 'push'], capture_output=True, text=True).stderr or 'pushed')
print(subprocess.run(['git', 'log', '--oneline', '-2'], capture_output=True, text=True).stdout)